# RobinReal — Live Demo

Hits the deployed `POST /listings` API and renders results as a ranked table + folium map.

**Before running:**
1. Start the harness: `cd 'challenge harness' && uv run uvicorn app.main:app --port 8000`
2. Start the tunnel: `./scripts/start_tunnel.sh` *(writes the public URL to `.api_base`)*
3. Run this notebook. `API_BASE` auto-loads from `.api_base`, falling back to `http://localhost:8000`.

To override manually: `API_BASE = 'https://…'` in the cell below, or `export ROBINREAL_API_BASE=https://…` before `jupyter lab`.

In [ ]:
%pip install -q requests folium pandas

In [ ]:
import os, requests, folium, pandas as pd
from pathlib import Path
from folium.features import DivIcon
from IPython.display import display, Markdown

def _load_api_base() -> str:
    """Prefer env var, then repo-root .api_base file (written by start_tunnel.sh),
    then localhost."""
    env = os.environ.get('ROBINREAL_API_BASE', '').strip()
    if env:
        return env
    api_file = Path(__file__).parent.parent / '.api_base' if '__file__' in globals() else Path('../.api_base')
    if api_file.exists():
        url = api_file.read_text().strip()
        if url:
            return url
    return 'http://localhost:8000'

API_BASE = _load_api_base()
print(f'API_BASE = {API_BASE}')

def search(query: str, limit: int = 25) -> dict:
    r = requests.post(f'{API_BASE}/listings', json={'query': query, 'limit': limit}, timeout=60)
    r.raise_for_status()
    return r.json()

In [ ]:
PIN_CSS = ('display:inline-block;padding:4px 10px;border-radius:16px;'
           'background:#fff;border:1.5px solid #222;box-shadow:0 1px 4px rgba(0,0,0,.25);'
           'font:600 12px/1 -apple-system,BlinkMacSystemFont,sans-serif;color:#222;white-space:nowrap;')
PIN_CSS_TOP = PIN_CSS.replace('#fff', '#111').replace('color:#222', 'color:#fff')

def _ranked_df(listings: list[dict]) -> pd.DataFrame:
    rows = []
    for item in listings:
        l = item.get('listing', {}) or {}
        rows.append({
            'listing_id': item.get('listing_id'),
            'score': item.get('score'),
            'title': l.get('title'),
            'city': l.get('city'),
            'price_chf': l.get('price_chf'),
            'rooms': l.get('rooms'),
            'reason': item.get('reason'),
        })
    return pd.DataFrame(rows)

def _popup_html(item: dict, rank: int) -> str:
    l = item.get('listing', {}) or {}
    img = l.get('hero_image_url') or (l.get('image_urls') or [None])[0]
    img_html = f'<img src="{img}" style="width:100%;border-radius:6px;margin-bottom:6px">' if img else ''
    price = f'CHF {l.get("price_chf"):,.0f}/mo' if l.get('price_chf') else 'price n/a'
    rooms = f'{l.get("rooms"):g} rooms' if l.get('rooms') else ''
    return (f'<div style="font:13px/1.4 sans-serif;width:220px">{img_html}'
            f'<div style="font-weight:700;margin-bottom:2px">#{rank} · {l.get("title","")}</div>'
            f'<div style="color:#555">{l.get("city") or ""} · {rooms}</div>'
            f'<div style="margin-top:4px"><b>{price}</b> · score {item.get("score",0):.2f}</div>'
            f'<div style="color:#666;margin-top:4px;font-size:12px">{item.get("reason","")}</div></div>')

def show_map(listings: list[dict], *, highlight_top: int = 1):
    pts = [i for i in listings if (i.get('listing') or {}).get('latitude') is not None]
    if not pts:
        print('(no coordinates to plot)'); return None
    lats = [i['listing']['latitude'] for i in pts]
    lons = [i['listing']['longitude'] for i in pts]
    m = folium.Map(location=[sum(lats)/len(lats), sum(lons)/len(lons)], zoom_start=12, tiles='CartoDB positron')
    for rank, item in enumerate(pts):
        l = item['listing']
        style = PIN_CSS_TOP if rank < highlight_top else PIN_CSS
        price = l.get('price_chf') or 0
        html = f'<div style="{style}">CHF {price:,.0f}</div>'
        folium.Marker(
            [l['latitude'], l['longitude']],
            icon=DivIcon(icon_size=(90, 24), icon_anchor=(45, 12), html=html),
            popup=folium.Popup(_popup_html(item, rank + 1), max_width=260),
            tooltip=f'#{rank + 1} · {l.get("title","")}',
        ).add_to(m)
    if len(pts) > 1:
        m.fit_bounds([[min(lats), min(lons)], [max(lats), max(lons)]], padding=(30, 30))
    return m

def demo(query: str, limit: int = 10) -> None:
    display(Markdown(f'### Query\n> {query}'))
    resp = search(query, limit=limit)
    listings = resp.get('listings', [])
    if not listings:
        display(Markdown('_No listings returned._')); return
    display(_ranked_df(listings))
    m = show_map(listings)
    if m is not None:
        display(m)

## Example queries

In [ ]:
demo('3.5-room bright apartment in Zurich under 2800 with balcony')

In [ ]:
demo('family friendly flat in Winterthur, not too expensive, ideally with parking')

In [ ]:
demo('modern quiet studio in Geneva with nice views')